# Employee Attrition Analysis — IBM HR Analytics Dataset

**Question:** why do employees leave, and which groups are most at risk?

**Data:** IBM HR Analytics Employee Attrition & Performance dataset (Kaggle).
1,470 employees, 35 columns. One row per employee.

**Honest note:** this dataset is simulated data created by IBM for education
and analysis practice. It is not a real company's workforce, so the findings
here are an analytical exercise — they show how I analyse attrition, not facts
about any real employer.

**Tools:** Python (pandas, matplotlib), SQL.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

In [ ]:
# Shape and data quality
print("Rows, columns:", df.shape)
print("Missing values in total:", df.isna().sum().sum())
print(df.dtypes.value_counts())

## 1. Overall attrition

Expected result from the published dataset: 237 of 1,470 employees left,
an attrition rate of 16.1%. 1,233 stayed (83.9%). The dataset has no
missing values.


In [ ]:
total = len(df)
left = (df["Attrition"] == "Yes").sum()
print(f"Employees: {total}")
print(f"Left: {left} ({left / total * 100:.1f}%)")
print(f"Stayed: {total - left} ({(total - left) / total * 100:.1f}%)")

## 2. Attrition by department

Which department loses the most people?


In [ ]:
dept = df.groupby("Department")["Attrition"].apply(
    lambda s: (s == "Yes").mean() * 100
).sort_values(ascending=False)
print(dept.round(1))
dept.plot(kind="bar", title="Attrition rate by department (%)")
plt.ylabel("Attrition rate (%)")
plt.tight_layout()
plt.show()

## 3. Attrition by job role

Same question, one level down — job roles inside each department.


In [ ]:
role = df.groupby("JobRole")["Attrition"].apply(
    lambda s: (s == "Yes").mean() * 100
).sort_values(ascending=False)
print(role.round(1))
role.plot(kind="barh", title="Attrition rate by job role (%)")
plt.xlabel("Attrition rate (%)")
plt.tight_layout()
plt.show()

## 4. Attrition by overtime

Does working overtime go with leaving? This is the split HR teams
usually ask about first.


In [ ]:
ot = df.groupby("OverTime")["Attrition"].apply(
    lambda s: (s == "Yes").mean() * 100
)
print(ot.round(1))
ot.plot(kind="bar", title="Attrition rate by overtime (%)")
plt.ylabel("Attrition rate (%)")
plt.tight_layout()
plt.show()

## 5. Attrition by years at the company

New joiners or long-serving staff — who leaves?


In [ ]:
bins = [0, 2, 5, 10, 20, 40]
labels = ["0-2 years", "3-5 years", "6-10 years", "11-20 years", "20+ years"]
df["TenureBand"] = pd.cut(df["YearsAtCompany"], bins=bins, labels=labels)
tenure = df.groupby("TenureBand", observed=True)["Attrition"].apply(
    lambda s: (s == "Yes").mean() * 100
)
print(tenure.round(1))
tenure.plot(kind="bar", title="Attrition rate by years at company (%)")
plt.ylabel("Attrition rate (%)")
plt.tight_layout()
plt.show()

## 6. Attrition by monthly-income band

Does pay level track with leaving?


In [ ]:
bins = [0, 3000, 5000, 8000, 12000, 20000]
labels = ["Under $3k", "$3k-5k", "$5k-8k", "$8k-12k", "Over $12k"]
df["IncomeBand"] = pd.cut(df["MonthlyIncome"], bins=bins, labels=labels)
inc = df.groupby("IncomeBand", observed=True)["Attrition"].apply(
    lambda s: (s == "Yes").mean() * 100
)
print(inc.round(1))
inc.plot(kind="bar", title="Attrition rate by monthly income band (%)")
plt.ylabel("Attrition rate (%)")
plt.tight_layout()
plt.show()

## 7. The income gap: leavers vs stayers

Average monthly income overall is $6,503. One published analysis of this
dataset found leavers earned $4,787 a month on average, against $6,832 for
stayers — a 42.7% gap. The cell below recomputes it from the data.


In [ ]:
gap = df.groupby("Attrition")["MonthlyIncome"].mean()
print(gap.round(0))
print("Overall average monthly income:", round(df["MonthlyIncome"].mean(), 0))
print("Gap (stayers - leavers):", round(gap["No"] - gap["Yes"], 0))

## 8. Attrition by performance rating — a check on a common assumption

A common assumption is that the people who leave are the low performers.
In one published dashboard built on this dataset, attrition was 16.4%
among employees rated Outstanding and 16.1% among those rated Excellent —
basically the same as the overall 16.1%. Leavers are not the weak
performers; the company is losing good people too.


In [ ]:
perf = df.groupby("PerformanceRating")["Attrition"].apply(
    lambda s: (s == "Yes").mean() * 100
)
print(perf.round(1))

## What this means — recommendations

1. **Look at overtime first.** If the overtime split is wide, overtime is
   the cheapest problem to fix: cap it, hire cover, or pay for it properly.
2. **Fix the pay gap for people who leave.** Leavers earn far less than
   stayers on average ($4,787 vs $6,832). Review pay for the roles and
   departments with the highest attrition before they walk out.
3. **Watch the first two years.** If new joiners leave fastest, onboarding
   and first-year manager support matter more than another retention bonus.
4. **Do not blame performance.** Attrition among top-rated employees is
   about the same as overall — this is a retention problem, not a
   performance problem.

## Limitations

- The dataset is simulated, so nothing here describes a real employer.
- These are associations, not causes: low pay and overtime travel together
  with leaving, but the data cannot prove one causes the other.
- One snapshot in time — no trend over years.
